# Real Recorded-Image Tin-Can Inspection Pipeline

This notebook processes **real recorded tin-can images** and joins them to **real weight readings from CSV**. It performs actual OpenCV measurements and EasyOCR recognition. Until a labelled dataset is available, visual rejection uses a transparent baseline anomaly model. After you train YOLOv5, place `best.pt` in `data/best.pt` (next to this notebook) and the same pipeline automatically uses it.

Important: pixel width/height are real image measurements, but they are not millimetres unless you add a calibration target. Weight comes only from your CSV; it is never invented. For the supplied synthetic starter set, `OCR_AFFECTS_DECISION=False` because OCR reliability must not be evaluated using computer-rendered placeholder labels. OCR still runs and is logged. Set it to `True` only for a validated real-image experiment.

**Local/VS Code version:** this notebook runs entirely locally — no Google Colab dependency. All inputs are read from, and outputs written to, a `data/` folder created next to this notebook the first time you run Cell 2.

In [2]:
# 1. Install the required modules (run once per environment/venv)
import sys, subprocess
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                 'easyocr', 'opencv-python-headless', 'pandas', 'matplotlib',
                 'seaborn', 'pyyaml', 'plotly'], check=True)

yolo_dir = Path('data/yolov5')
if not yolo_dir.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/ultralytics/yolov5', str(yolo_dir)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(yolo_dir / 'requirements.txt')], check=True)
print('Installation complete')

FileNotFoundError: [WinError 2] The system cannot find the file specified

In [ ]:
# 2. Configuration
from pathlib import Path
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import easyocr, torch, re, time, json, zipfile, shutil, warnings
from datetime import datetime, date
from IPython.display import display, Image as IPImage, HTML
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Ellipse

# Local working directory - all paths live under data/, created next to this notebook
DATA_ROOT = Path.cwd() / 'data'
IMAGE_DIR = DATA_ROOT / 'input_images'
GOOD_DIR = DATA_ROOT / 'good_reference'   # known-good cans photographed with same setup
WEIGHT_CSV = DATA_ROOT / 'weights.csv'
YOLO_DIR = DATA_ROOT / 'yolov5'
YOLO_WEIGHTS = DATA_ROOT / 'best.pt'
OUTPUT_DIR = DATA_ROOT / 'inspection_output'
ANNOTATED_DIR = OUTPUT_DIR / 'annotated'
for p in [IMAGE_DIR, GOOD_DIR, OUTPUT_DIR, ANNOTATED_DIR]: p.mkdir(parents=True, exist_ok=True)

NOMINAL_WEIGHT_G = 400.0
WEIGHT_TOLERANCE_G = 12.0
ANOMALY_THRESHOLD = 4.0
YOLO_CONFIDENCE = 0.35
OCR_MIN_CONFIDENCE = 0.30
REQUIRE_EXPIRY_DATE = True
OCR_AFFECTS_DECISION = False  # False for synthetic samples; set True for validated real-image experiments
DEFECT_CLASSES = {'dent', 'scratch', 'rust', 'deformation', 'damaged'}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
print('Configuration ready. Data root:', DATA_ROOT.resolve())

## Input files

Place these in the local `data/` folder next to this notebook (created automatically when you run Cell 2):

1. `data/input_images/` containing the cans to inspect, or a `data/input_images_12pass_6reject.zip` for Cell 4 to extract automatically.
2. `data/good_reference/` containing 30 verified synthetic good references, or `data/good_reference_12pass_6reject.zip`.
3. `data/weights.csv` with exactly `image_name,weight_g`. Example: `can_001.jpg,398.6`.
4. Optionally `data/best.pt` after YOLOv5 training.

The image filename is the key that connects each photograph to its weight reading.

In [ ]:
# 3. Extract local inputs (if zipped) and check the weights CSV
def extract_zip(zip_path, destination):
    if zip_path.exists():
        shutil.rmtree(destination, ignore_errors=True); destination.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as archive: archive.extractall(destination)
        print(f'Extracted {zip_path.name} -> {destination}')

extract_zip(DATA_ROOT / 'input_images_12pass_6reject.zip', IMAGE_DIR)
extract_zip(DATA_ROOT / 'good_reference_12pass_6reject.zip', GOOD_DIR)

if not WEIGHT_CSV.exists():
    template = pd.DataFrame({'image_name':['can_001.jpg','can_002.jpg'], 'weight_g':[400.0,398.5]})
    template_path = DATA_ROOT / 'weights_template.csv'
    template.to_csv(template_path, index=False)
    print(f'{WEIGHT_CSV.name} missing. A template was written to {template_path.resolve()} - '
          f'fill it in and save it as {WEIGHT_CSV.resolve()}.')
else:
    print('Inputs ready in', DATA_ROOT.resolve())

In [ ]:
# 4. Validate inputs - stops instead of generating fake values
def image_files(folder):
    return sorted(p for p in folder.rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS)

images = image_files(IMAGE_DIR)
good_images = image_files(GOOD_DIR)
if not images: raise FileNotFoundError('No test images found in input_images.zip')
if not WEIGHT_CSV.exists(): raise FileNotFoundError('Place weights.csv in the data folder and run the previous cell again')
weights = pd.read_csv(WEIGHT_CSV)
required = {'image_name','weight_g'}
if not required.issubset(weights.columns): raise ValueError('weights.csv requires image_name and weight_g columns')
weights['image_name'] = weights['image_name'].astype(str).map(lambda x: Path(x).name)
weights['weight_g'] = pd.to_numeric(weights['weight_g'], errors='coerce')
if weights.image_name.duplicated().any(): raise ValueError('Duplicate image_name values found in weights.csv')
missing_weights = sorted({p.name for p in images} - set(weights.image_name))
if missing_weights: raise ValueError(f'Missing CSV weights for: {missing_weights[:10]}')
print(f'{len(images)} inspection images | {len(good_images)} known-good reference images | {len(weights)} weights')

In [ ]:
# 5. Actual OpenCV segmentation and measurements
FEATURE_COLUMNS = ['aspect_ratio','solidity','extent','edge_density','rust_ratio']

def segment_can(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (7,7), 0)
    _, a = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    b = cv2.bitwise_not(a)
    candidates = []
    for mask in [a,b]:
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9,9),np.uint8))
        contours,_ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            area = cv2.contourArea(contour)
            if .05*image.shape[0]*image.shape[1] < area < .95*image.shape[0]*image.shape[1]:
                candidates.append((area, contour, mask))
    if not candidates: raise ValueError('Can contour not found; use a plain contrasting background')
    _, contour, mask = max(candidates, key=lambda item:item[0])
    clean = np.zeros_like(mask); cv2.drawContours(clean,[contour],-1,255,-1)
    return contour, clean

def measure_image(path):
    image = cv2.imread(str(path))
    if image is None: raise ValueError(f'Cannot read {path}')
    contour, mask = segment_can(image)
    x,y,w,h = cv2.boundingRect(contour); area = cv2.contourArea(contour)
    hull_area = cv2.contourArea(cv2.convexHull(contour))
    roi_area = max(w*h,1)
    edges = cv2.Canny(cv2.cvtColor(image,cv2.COLOR_BGR2GRAY),80,170)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    rust = cv2.inRange(hsv, np.array([0,60,35]), np.array([25,255,210]))
    can_pixels = max(np.count_nonzero(mask),1)
    return {
      'width_px':w, 'height_px':h, 'area_px':area, 'aspect_ratio':w/max(h,1),
      'solidity':area/max(hull_area,1), 'extent':area/roi_area,
      'edge_density':np.count_nonzero(cv2.bitwise_and(edges,edges,mask=mask))/can_pixels,
      'rust_ratio':np.count_nonzero(cv2.bitwise_and(rust,rust,mask=mask))/can_pixels,
      '_image':image, '_mask':mask, '_bbox':(x,y,w,h)
    }

In [ ]:
# 6. Build the known-good baseline (robust median and MAD)
reference_source = good_images
if len(reference_source) < 10:
    warnings.warn('Fewer than 10 known-good references. Results are preliminary, not a validated defect model.')
if not reference_source:
    raise ValueError('Add known-good images to good_reference.zip; the system will not invent a baseline')
reference_rows = []
for path in reference_source:
    try:
        m = measure_image(path); reference_rows.append({k:m[k] for k in FEATURE_COLUMNS})
    except Exception as e: print('Reference skipped:', path.name, e)
reference_df = pd.DataFrame(reference_rows)
if len(reference_df) < 5: raise ValueError('At least 5 segmentable known-good reference images are required')
baseline_median = reference_df.median()
baseline_mad = (reference_df-baseline_median).abs().median().replace(0,1e-6)
display(pd.DataFrame({'median':baseline_median,'MAD':baseline_mad}))

In [ ]:
# 7. Load real OCR and optional trained YOLOv5 model
ocr_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
yolo_model = None
if YOLO_WEIGHTS.exists():
    yolo_model = torch.hub.load(str(YOLO_DIR),'custom',path=str(YOLO_WEIGHTS),source='local')
    yolo_model.conf = YOLO_CONFIDENCE
    print('YOLOv5 mode enabled:', YOLO_WEIGHTS)
else:
    print('Baseline anomaly mode enabled. Train YOLOv5 before claiming supervised defect classification.')

DATE_PATTERNS = [r'\b(\d{4}[-/.]\d{1,2}[-/.]\d{1,2})\b', r'\b(\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4})\b']
def parse_expiry(text):
    for pattern in DATE_PATTERNS:
        match = re.search(pattern,text)
        if match:
            parsed = pd.to_datetime(match.group(1),errors='coerce',dayfirst=True)
            if not pd.isna(parsed): return parsed.date()
    return None

def run_ocr(image):
    items = ocr_reader.readtext(image,detail=1,paragraph=False)
    accepted = [(text,float(conf)) for _,text,conf in items if conf >= OCR_MIN_CONFIDENCE]
    text = ' '.join(item[0] for item in accepted)
    confidence = np.mean([item[1] for item in accepted]) if accepted else 0.0
    return text, float(confidence), parse_expiry(text)

def run_yolo(image):
    if yolo_model is None: return [], 0.0
    detections = yolo_model(image).pandas().xyxy[0]
    names = [str(x).lower() for x in detections.name.tolist()]
    confidence = float(detections.confidence.max()) if len(detections) else 0.0
    return names, confidence

In [ ]:
# 8. Process every real image and save auditable annotated results
weight_map = weights.set_index('image_name').weight_g.to_dict()
records = []
for number,path in enumerate(images,1):
    started = time.perf_counter()
    try:
        m = measure_image(path); image = m.pop('_image'); mask=m.pop('_mask'); bbox=m.pop('_bbox')
        robust_z = ((pd.Series({k:m[k] for k in FEATURE_COLUMNS})-baseline_median).abs()/(1.4826*baseline_mad)).replace([np.inf],np.nan).fillna(0)
        anomaly_score = float(robust_z.max())
        anomalous_features = ','.join(robust_z[robust_z>ANOMALY_THRESHOLD].index) or 'none'
        ocr_text,ocr_conf,expiry = run_ocr(image)
        classes,yolo_conf = run_yolo(image)
        detected_defects = sorted(set(classes)&DEFECT_CLASSES)
        visual_ok = not detected_defects if yolo_model is not None else anomaly_score <= ANOMALY_THRESHOLD
        weight_g = float(weight_map[path.name])
        weight_ok = abs(weight_g-NOMINAL_WEIGHT_G)<=WEIGHT_TOLERANCE_G
        ocr_ok = bool(ocr_text)
        expiry_ok = expiry is not None and expiry >= date.today() if REQUIRE_EXPIRY_DATE else True
        reasons=[]
        if not visual_ok: reasons.append('visual:'+(','.join(detected_defects) if detected_defects else anomalous_features))
        if not weight_ok: reasons.append('weight')
        if OCR_AFFECTS_DECISION and not ocr_ok: reasons.append('ocr_unreadable')
        if OCR_AFFECTS_DECISION and not expiry_ok: reasons.append('expiry_missing_or_expired')
        decision='PASS' if not reasons else 'REJECT'
        x,y,w,h=bbox; color=(0,180,0) if decision=='PASS' else (0,0,230)
        annotated=image.copy(); cv2.rectangle(annotated,(x,y),(x+w,y+h),color,3)
        cv2.putText(annotated,f'{decision} | {path.name}',(10,30),cv2.FONT_HERSHEY_SIMPLEX,.75,color,2)
        cv2.putText(annotated,'; '.join(reasons) if reasons else 'all checks passed',(10,60),cv2.FONT_HERSHEY_SIMPLEX,.55,color,2)
        cv2.imwrite(str(ANNOTATED_DIR/path.name),annotated)
        records.append({'image_name':path.name,**m,'anomaly_score':anomaly_score,'anomalous_features':anomalous_features,
          'yolo_classes':','.join(classes) or 'not_used','yolo_confidence':yolo_conf,'ocr_text':ocr_text,'ocr_confidence':ocr_conf,
          'ocr_readable':ocr_ok,'expiry_detected':expiry.isoformat() if expiry else None,'expiry_valid':expiry_ok,
          'ocr_affects_decision':OCR_AFFECTS_DECISION,'weight_g':weight_g,'weight_ok':weight_ok,
          'decision':decision,'reject_reason':'; '.join(reasons) or 'none','processing_ms':(time.perf_counter()-started)*1000})
        print(f'[{number}/{len(images)}] {path.name}: {decision}')
    except Exception as e:
        records.append({'image_name':path.name,'decision':'ERROR','reject_reason':str(e),'processing_ms':(time.perf_counter()-started)*1000})
results=pd.DataFrame(records)
results.to_csv(OUTPUT_DIR/'inspection_results.csv',index=False)
display(results)

In [ ]:
# 9. Conveyor simulation driven by the REAL inspection results
animation_rows = results[results.decision.isin(['PASS','REJECT'])].reset_index(drop=True)
if animation_rows.empty: raise ValueError('No successful inspection results are available for animation')
FRAMES_PER_CAN = 10
fig,ax=plt.subplots(figsize=(13,6.3)); fig.patch.set_facecolor('#dce3e7')

def animated_can(x,y,state,label):
    body='#b9c3c8' if state=='SCANNING' else ('#61b66b' if state=='PASS' else '#ef655c')
    ax.add_patch(Ellipse((x+.07,y-.06),.72,.16,color='black',alpha=.20,zorder=8))
    ax.add_patch(plt.Rectangle((x-.32,y),.64,.95,facecolor=body,edgecolor='#455a64',lw=1.2,zorder=10))
    ax.add_patch(Ellipse((x,y+.95),.64,.17,facecolor='#edf1f3',edgecolor='#455a64',zorder=12))
    ax.add_patch(Ellipse((x,y),.64,.15,facecolor='#89979e',edgecolor='#455a64',zorder=11))
    ax.add_patch(plt.Rectangle((x-.28,y+.27),.56,.40,facecolor='#176b3a',zorder=11))
    ax.text(x,y+.47,label[-7:],ha='center',va='center',fontsize=6,color='white',weight='bold',zorder=13)

def conveyor_frame(frame):
    ax.clear(); ax.set_xlim(0,13); ax.set_ylim(0,6.3); ax.axis('off'); ax.set_facecolor('#edf1f3')
    ax.add_patch(plt.Rectangle((0,0),13,1.65,color='#bdc8cd'))
    ax.text(.3,5.95,'MULTIMODAL AI TIN-CAN INSPECTION',fontsize=15,weight='bold',color='#263238')
    # Belt and moving steel slats
    ax.add_patch(plt.Polygon([[.45,1.95],[8.65,1.95],[9.05,2.90],[.85,2.90]],closed=True,facecolor='#263238',edgecolor='#101719',lw=2,zorder=3))
    offset=(frame%7)/7
    for sx in np.arange(.6,8.8,.48):
        px=sx+offset*.48; ax.plot([px,px+.34],[2.0,2.84],color='#607d8b',lw=1.2,zorder=4)
    ax.plot([.5,8.9],[1.95,1.95],color='#90a4ae',lw=7,zorder=5)
    for sx in np.arange(.85,8.8,1.0): ax.add_patch(plt.Circle((sx,1.95),.19,fc='#b0bec5',ec='#455a64',zorder=6))
    for sx in [1.0,3.5,6.0,8.5]:
        ax.plot([sx,sx-.15],[1.85,.55],color='#607d8b',lw=6); ax.plot([sx-.38,sx+.12],[.53,.53],color='#455a64',lw=6)
    # Inspection gantry, camera beam and load cell
    for gx in [4.75,6.25]: ax.plot([gx,gx],[2.02,5.15],color='#607d8b',lw=8,zorder=7)
    ax.plot([4.75,6.25],[5.12,5.12],color='#455a64',lw=9,zorder=7)
    ax.add_patch(plt.Rectangle((5.18,4.52),.65,.40,fc='#17252b',ec='#90a4ae',lw=2,zorder=9))
    ax.add_patch(plt.Circle((5.50,4.50),.14,fc='#42a5f5',ec='white',zorder=10))
    ax.add_patch(plt.Polygon([[5.35,4.36],[5.65,4.36],[6.0,2.84],[5.0,2.84]],color='#29b6f6',alpha=.18,zorder=6))
    ax.add_patch(plt.Rectangle((6.42,2.20),1.05,.12,fc='#90caf9',ec='#1565c0',zorder=7))
    ax.text(6.95,1.76,'WEIGHT SENSOR',ha='center',fontsize=7,weight='bold',color='#0d47a1')
    # Pusher, chutes and bins
    ax.add_patch(plt.Rectangle((7.82,3.58),.65,.25,fc='#37474f',ec='#111',zorder=8))
    ax.plot([8.65,10.25],[2.72,4.03],color='#607d8b',lw=12,zorder=3)
    ax.plot([8.65,10.25],[2.25,1.08],color='#607d8b',lw=12,zorder=3)
    ax.add_patch(plt.Rectangle((10.1,3.18),2.35,1.18,fc='#2e7d32',ec='#1b5e20',lw=2,zorder=4))
    ax.add_patch(plt.Rectangle((10.1,.25),2.35,1.18,fc='#c62828',ec='#8e0000',lw=2,zorder=4))
    ax.text(11.27,3.77,'ACCEPTED',ha='center',va='center',color='white',weight='bold',zorder=5)
    ax.text(11.27,.84,'REJECTED',ha='center',va='center',color='white',weight='bold',zorder=5)
    # Current inspected product
    idx=min(frame//FRAMES_PER_CAN,len(animation_rows)-1); phase=(frame%FRAMES_PER_CAN)/(FRAMES_PER_CAN-1)
    row=animation_rows.iloc[idx]; inspected=phase>=.54
    if phase<.72: x=.80+(phase/.72)*7.60; y=2.82
    else:
        t=(phase-.72)/.28; x=8.40+1.70*t; y=2.82+(1.02*t if row.decision=='PASS' else -1.92*t)
    state=row.decision if inspected else 'SCANNING'; animated_can(x,y,state,row.image_name)
    push=max(0,1-abs(phase-.74)/.09) if row.decision=='REJECT' else 0
    ax.plot([8.14,8.14],[3.57,3.57-push*.65],color='#ffb300',lw=7,zorder=12)
    ax.add_patch(plt.Rectangle((7.85,3.45-push*.65),.58,.13,fc='#ffca28',ec='#6d4c41',zorder=13))
    # Live result panel and current recorded photograph
    ax.add_patch(plt.Rectangle((.25,3.58),4.05,1.82,fc='#15313b',ec='#78909c',lw=2,zorder=14))
    result_color='#66bb6a' if row.decision=='PASS' else '#ef5350'
    ax.text(.45,5.08,f'IMAGE {idx+1:02d}/{len(animation_rows):02d}: {row.image_name}',color='#80d8ff',fontsize=9,weight='bold',zorder=15)
    ax.text(.45,4.70,f'WEIGHT: {row.weight_g:.1f} g',color='white',fontsize=9,zorder=15)
    ax.text(.45,4.35,f'OCR: {str(row.ocr_text)[:30]}',color='white',fontsize=8,zorder=15)
    ax.text(.45,3.98,f'REASON: {row.reject_reason}',color='white',fontsize=8,zorder=15)
    ax.text(3.45,4.50,row.decision if inspected else 'SCANNING',ha='center',color=result_color,fontsize=12,weight='bold',zorder=15)

simulation=FuncAnimation(fig,conveyor_frame,frames=len(animation_rows)*FRAMES_PER_CAN,interval=140,repeat=False)
plt.close(fig); display(HTML(simulation.to_jshtml()))

In [ ]:
# 10. Interactive 3D conveyor simulation driven by the REAL inspection results
import plotly.graph_objects as go

rows3d = results[results.decision.isin(['PASS','REJECT'])].reset_index(drop=True)
if rows3d.empty: raise ValueError('No successful inspection results are available for 3D simulation')

def box3d(x0,x1,y0,y1,z0,z1,color,name,opacity=1.0):
    x=[x0,x1,x1,x0,x0,x1,x1,x0]; y=[y0,y0,y1,y1,y0,y0,y1,y1]; z=[z0,z0,z0,z0,z1,z1,z1,z1]
    i=[0,0,0,1,1,2,4,4,5,6,3,2]; j=[1,2,4,2,5,3,5,6,6,7,7,6]; k=[2,4,1,5,2,7,6,7,1,3,2,7]
    return go.Mesh3d(x=x,y=y,z=z,i=i,j=j,k=k,color=color,opacity=opacity,name=name,hoverinfo='name',flatshading=True)

def can3d(cx,cy,cz,color,hover):
    theta=np.linspace(0,2*np.pi,18); height=np.linspace(0,.95,8)
    T,H=np.meshgrid(theta,height); radius=.24
    X=cx+radius*np.cos(T); Y=cy+radius*np.sin(T); Z=cz+H
    return go.Surface(x=X,y=Y,z=Z,surfacecolor=np.zeros_like(Z),colorscale=[[0,color],[1,color]],cmin=0,cmax=1,showscale=False,name='Current can',hovertemplate=hover+'<extra></extra>',lighting=dict(ambient=.55,diffuse=.8,specular=.65,roughness=.3))

static3d=[
    box3d(.3,8.6,-.75,.75,.40,.72,'#263238','Conveyor belt'),
    box3d(.2,8.7,-.88,-.72,.28,.85,'#78909c','Left rail'),
    box3d(.2,8.7,.72,.88,.28,.85,'#78909c','Right rail'),
    box3d(4.15,4.35,-1.05,1.05,.35,3.20,'#607d8b','Inspection gantry'),
    box3d(5.55,5.75,-1.05,1.05,.35,3.20,'#607d8b','Inspection gantry'),
    box3d(4.15,5.75,-1.05,1.05,3.05,3.25,'#455a64','Gantry beam'),
    box3d(4.72,5.18,-.32,.32,2.55,3.02,'#1565c0','Vision/OCR camera'),
    box3d(6.0,7.1,-.68,.68,.73,.82,'#42a5f5','Weight sensor'),
    box3d(9.7,11.3,1.15,2.75,.10,1.20,'#2e7d32','Accepted bin',.90),
    box3d(9.7,11.3,-2.75,-1.15,.10,1.20,'#c62828','Rejected bin',.90),
    go.Scatter3d(x=[8.3,10.0],y=[0,1.85],z=[.72,1.10],mode='lines',line=dict(color='#43a047',width=16),name='Pass chute'),
    go.Scatter3d(x=[8.3,10.0],y=[0,-1.85],z=[.72,.50],mode='lines',line=dict(color='#e53935',width=16),name='Reject chute'),
    go.Cone(x=[4.95],y=[0],z=[2.55],u=[0],v=[0],w=[-1.7],sizemode='absolute',sizeref=1.2,anchor='tail',colorscale=[[0,'#29b6f6'],[1,'#81d4fa']],showscale=False,opacity=.25,name='Scanning beam')
]

FRAMES_3D=8; frames3d=[]
for idx,row in rows3d.iterrows():
    for step in range(FRAMES_3D):
        phase=step/(FRAMES_3D-1)
        if phase<.72:
            cx=.65+(phase/.72)*7.65; cy=0.; cz=.74
        else:
            t=(phase-.72)/.28; cx=8.30+1.75*t; cy=(1.85 if row.decision=='PASS' else -1.85)*t; cz=.74+(.30 if row.decision=='PASS' else -.22)*t
        scanned=phase>=.50; state=row.decision if scanned else 'SCANNING'
        color='#43a047' if state=='PASS' else ('#e53935' if state=='REJECT' else '#b0bec5')
        hover=f'<b>{row.image_name}</b><br>Status: {state}<br>Weight: {row.weight_g:.1f} g<br>OCR: {str(row.ocr_text)[:35]}<br>Reason: {row.reject_reason}'
        title=f'3D Multimodal Inspection | {idx+1}/{len(rows3d)} | {row.image_name} | {state}'
        frames3d.append(go.Frame(data=[can3d(cx,cy,cz,color,hover)],traces=[len(static3d)],name=f'{idx}-{step}',layout=go.Layout(title_text=title)))

fig3d=go.Figure(data=static3d+[frames3d[0].data[0]],frames=frames3d)
fig3d.update_layout(title='3D Multimodal Tin-Can Inspection',height=720,showlegend=True,
    scene=dict(xaxis=dict(title='Production direction',range=[0,11.5],showbackground=True,backgroundcolor='#e9eef1'),yaxis=dict(title='Sorting direction',range=[-3.2,3.2],showbackground=True,backgroundcolor='#dfe7ea'),zaxis=dict(title='Height',range=[0,3.6],showbackground=True,backgroundcolor='#f4f6f7'),aspectmode='manual',aspectratio=dict(x=2.4,y=1.25,z=.8),camera=dict(eye=dict(x=1.55,y=-1.65,z=1.15))),
    updatemenus=[dict(type='buttons',showactive=False,x=.05,y=1.08,direction='left',buttons=[dict(label='Play 3D',method='animate',args=[None,{'frame':{'duration':180,'redraw':True},'transition':{'duration':50},'fromcurrent':True}]),dict(label='Pause',method='animate',args=[[None],{'frame':{'duration':0,'redraw':False},'mode':'immediate'}])])],margin=dict(l=0,r=0,b=0,t=70))
fig3d.show()

In [ ]:
# 11. Manufacturing analytics and visual evidence
valid=results[results.decision!='ERROR'].copy()
fig,axes=plt.subplots(1,3,figsize=(16,4.5))
valid.decision.value_counts().plot(kind='bar',ax=axes[0],color=['#2e7d32','#c62828'],title='Inspection Decisions')
sns.histplot(valid.weight_g,kde=True,ax=axes[1],color='#1565c0'); axes[1].axvline(NOMINAL_WEIGHT_G-WEIGHT_TOLERANCE_G,c='r',ls='--'); axes[1].axvline(NOMINAL_WEIGHT_G+WEIGHT_TOLERANCE_G,c='r',ls='--'); axes[1].set_title('Measured CSV Weights')
sns.histplot(valid.anomaly_score,kde=True,ax=axes[2],color='#6a1b9a'); axes[2].axvline(ANOMALY_THRESHOLD,c='r',ls='--'); axes[2].set_title('Visual Anomaly Scores')
plt.tight_layout(); plt.show()
print('Mean processing time:',round(valid.processing_ms.mean(),1),'ms/image')
print('Throughput:',round(1000/valid.processing_ms.mean(),2),'images/second')

samples=list(ANNOTATED_DIR.glob('*'))[:6]
fig,axes=plt.subplots(2,3,figsize=(15,9)); axes=axes.ravel()
for ax in axes: ax.axis('off')
for ax,path in zip(axes,samples):
    ax.imshow(cv2.cvtColor(cv2.imread(str(path)),cv2.COLOR_BGR2RGB)); ax.set_title(path.name); ax.axis('off')
plt.tight_layout(); plt.show()

## YOLOv5 training stage (run after annotation)

Collect images from the final camera position and label visible defects such as `dent`, `scratch`, `rust` and `deformation` in YOLO format. Split independent images into train, validation and test sets. Your dataset ZIP should contain `data.yaml`, `images/train`, `images/val`, `labels/train`, and `labels/val`. Do not report YOLO results until you evaluate the trained model on a held-out test set.

In [ ]:
# 12. Upload a YOLO-format dataset ZIP locally, then train YOLOv5s
# Change RUN_TRAINING to True only after you have labelled data.
RUN_TRAINING=False
if RUN_TRAINING:
    dataset_zip=DATA_ROOT/'tin_can_yolo_dataset.zip'
    dataset_dir=DATA_ROOT/'tin_can_dataset'
    if not dataset_zip.exists():
        raise FileNotFoundError(f'Place tin_can_yolo_dataset.zip in {DATA_ROOT.resolve()} before training')
    shutil.rmtree(dataset_dir,ignore_errors=True); dataset_dir.mkdir()
    with zipfile.ZipFile(dataset_zip) as archive: archive.extractall(dataset_dir)
    data_yaml=next(dataset_dir.rglob('data.yaml'))
    subprocess.run([sys.executable, str(YOLO_DIR/'train.py'),
                     '--img','640','--batch','16','--epochs','100',
                     '--data',str(data_yaml),'--weights','yolov5s.pt',
                     '--project',str(DATA_ROOT/'yolo_runs'),'--name','tin_can_defects'], check=True)
    trained=DATA_ROOT/'yolo_runs'/'tin_can_defects'/'weights'/'best.pt'
    shutil.copy(trained,YOLO_WEIGHTS)
    print('Training complete. Rerun from Cell 7 to enable trained YOLO mode.')
else:
    print('Training skipped until labelled data is available.')

In [ ]:
# 13. Package all evidence for dissertation analysis
archive_path=shutil.make_archive(str(DATA_ROOT/'tin_can_inspection_evidence'),'zip',OUTPUT_DIR)
print('Created:', Path(archive_path).resolve())

## Experimental validity checklist

- Keep camera distance, angle, background and illumination fixed.
- Use at least 20-50 verified good cans for the initial baseline.
- Record the exact image-to-weight timestamp or filename mapping.
- Calibrate pixels to millimetres using a reference object if physical dimensions are reported.
- Label a diverse defect dataset and retain an independent test set.
- Report precision, recall, F1, mAP, OCR accuracy, weight error, processing latency and confusion matrix.
- Treat baseline-mode output as anomaly screening, not YOLO defect classification.